# Project Overview

The Ontario education curriculum is comprised of a series of documents that cover the following topics:

- Kindergarten
- The Arts
- French as a Second Language
- Health and Physical Education
- Language
- Mathematics
- Native Languages
- Science and Technology
- Social Studies, History, and Geography

Each core subject has a separate PDF document that covers all of the specific requirements for that particular grade. Teachers have to become very familiar with these documents as these are the guidelines to which they should be teaching for the school year.

To this end, the purpose of this project is to create a chatbot that leverages the power of Generative AI and LLMs to allow teachers to pose questions about the curriculum using natural language, and to receive distilled, summarized results, based on the latest curriculum. Only the Mathematics and Language results are used in this proof of concept. However, this can be easily extended to cover any number of curriculum topics.

In [1]:
!pip install -q -U langchain langchain-community langchain-core langchain-text-splitters langchain-openai langchain-chroma pypdf

In [2]:
import os
from pathlib import Path

DATA_PATH = "C:/Users/rossi/OneDrive/Data Science Projects/ON Edu RAG Project/"
CHROMA_PATH = "./chroma_db" 
COLLECTION_NAME = "ontario_education_docs"
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

# --- ENVIRONMENT SETUP ---

if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key:  ········


In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def load_and_split_documents(data_path: str) -> list:
    """Loads PDFs from the specified directory and splits them into chunks."""
    print(f"Loading PDFs from: {data_path}")
    loader = PyPDFDirectoryLoader(data_path)
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} source pages.")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=500,
        length_function=len,
        add_start_index=True
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split pages into {len(chunks)} text chunks.")
    return chunks

def build_or_load_vector_store(chunks: list, store_path: str, embedding_model) -> Chroma:
    """Creates a local Chroma DB collection if it doesn't exist, or loads it if it does."""
    if Path(store_path).exists():
        print(f"Loading existing vector database from: {store_path}")
        return Chroma(
            persist_directory=store_path,
            embedding_function=embedding_model,
            collection_name=COLLECTION_NAME
        )
    else:
        print(f"Creating new Chroma vector database at: {store_path}")
        return Chroma.from_documents(
            documents=chunks,
            embedding=embedding_model,
            persist_directory=store_path,
            collection_name=COLLECTION_NAME
        )

def get_rag_chain(llm):
    """Defines and binds the LCEL RAG execution assembly line."""
    prompt_template = PromptTemplate.from_template("""
Answer the question based only on the following context:

{context}

___

Answer the question based on the above context: {query}
""")
    output_parser = StrOutputParser()
    return prompt_template | llm | output_parser

C:\Users\rossi\AppData\Local\Temp\ipykernel_10916\1053160803.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [4]:
#Execute Pipeline

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

chunks = load_and_split_documents(DATA_PATH)
db = build_or_load_vector_store(chunks, CHROMA_PATH, embeddings)

rag_chain = get_rag_chain(llm)

Loading PDFs from: C:/Users/rossi/OneDrive/Data Science Projects/ON Edu RAG Project/
Successfully loaded 250 source pages.
Split pages into 841 text chunks.
Creating new Chroma vector database at: ./chroma_db


In [5]:
#Running RAG query

query_text = "What is the big idea that students are supposed to learn from the Grade 3 Language curriculum"

results = db.similarity_search_with_relevance_scores(query_text, k=3)
context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])

response_text = rag_chain.invoke({
    "context": context_text,
    "query": query_text
})

print(f"\n=== ANSWER ===\n{response_text}")


=== ANSWER ===
The big idea that students are supposed to learn from the Grade 3 Language curriculum is the integration and application of transferable skills—such as critical thinking, communication, collaboration, and digital literacy—across various language and literacy contexts. This includes understanding how these skills support effective communication, enhance engagement in learning, and foster an appreciation of diverse identities and perspectives, particularly in relation to cultural and community contexts. Additionally, students are encouraged to apply their learning across different subject areas and recognize the relevance of these skills in everyday life.
